# Версия v1: производные признаки просрочки

Цель — проверить, добавляют ли два понятных производных признака
устойчивый прирост качества по сравнению с версией v0.

Сравнение выполняется на той же кросс-валидации, с тем же
RANDOM_STATE, той же нагрузкой 10% и той же функцией оценки.

Тестовая выборка не используется.

In [ ]:
import pandas as pd
import numpy as np

from google.colab import files
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

TARGET = "default payment next month"
RANDOM_STATE = 42
N_SPLITS = 5
TOP_SHARE = 0.10

In [ ]:
uploaded = files.upload()

In [ ]:
train = pd.read_csv("train.csv")

train.shape

(24000, 25)

In [ ]:
train_summary = pd.Series({
    "Клиентов": len(train),
    "Событий неплатежа": train[TARGET].sum(),
    "Доля событий, %": train[TARGET].mean() * 100,
    "Уникальных ID": train["ID"].nunique()
})

train_summary

Клиентов             24000.000000
Событий неплатежа     5309.000000
Доля событий, %         22.120833
Уникальных ID        24000.000000
dtype: float64

In [ ]:
payment_status_columns = [
    "PAY_0",
    "PAY_2",
    "PAY_3",
    "PAY_4",
    "PAY_5",
    "PAY_6"
]

bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

train["TOTAL_BILL_AMT"] = train[bill_columns].sum(axis=1)
train["TOTAL_PAY_AMT"] = train[payment_columns].sum(axis=1)
train["NUM_DELINQ"] = (
    train[payment_status_columns] > 0
).sum(axis=1)

In [ ]:
train["MAX_DELINQ"] = train[payment_status_columns].max(axis=1)
train["RECENT_DELINQ_FLAG"] = (
    train[payment_status_columns[:3]] > 0
).any(axis=1).astype(int)

In [ ]:
base_feature_columns = [
    "LIMIT_BAL",
    "AGE",
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    *payment_status_columns,
    *bill_columns,
    *payment_columns,
    "TOTAL_BILL_AMT",
    "TOTAL_PAY_AMT",
    "NUM_DELINQ"
]

extended_feature_columns = [
    *base_feature_columns,
    "MAX_DELINQ",
    "RECENT_DELINQ_FLAG"
]

categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE",
    *payment_status_columns
]

len(base_feature_columns), len(extended_feature_columns)

(26, 28)

In [ ]:
def build_model(feature_columns):
    categorical = [
        column
        for column in categorical_features
        if column in feature_columns
    ]

    numeric = [
        column
        for column in feature_columns
        if column not in categorical
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                StandardScaler(),
                numeric
            ),
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical
            )
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            (
                "logistic_regression",
                LogisticRegression(
                    C=1.0,
                    max_iter=2000,
                    solver="lbfgs"
                )
            )
        ]
    )

In [ ]:
def evaluate_top_share(
    data,
    target_column,
    scores,
    share=0.10,
    method_name=""
):
    selected_count = int(np.ceil(len(data) * share))

    ranked = (
        data
        .assign(score=scores)
        .sort_values(
            ["score", "ID"],
            ascending=[False, True]
        )
        .head(selected_count)
    )

    y = ranked[target_column]
    base_rate = data[target_column].mean()

    return {
        "Метод": method_name,
        "Нагрузка, %": len(ranked) / len(data) * 100,
        "Событий в отборе": int(y.sum()),
        "Доля событий, %": y.mean() * 100,
        "Recall, %": y.sum() / data[target_column].sum() * 100,
        "Lift": y.mean() / base_rate
    }

In [ ]:
X = train[extended_feature_columns]
y = train[TARGET]

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_results = []

for fold, (fit_index, valid_index) in enumerate(cv.split(X, y), start=1):
    X_fold_fit = X.iloc[fit_index]
    y_fold_fit = y.iloc[fit_index]

    X_fold_valid = X.iloc[valid_index]

    fold_predictions = train.loc[
        X_fold_valid.index,
        ["ID", TARGET, "PAY_0"]
    ].copy()

    for version_name, columns in [
        ("v0 baseline", base_feature_columns),
        ("v1 extended", extended_feature_columns)
    ]:
        fold_model = build_model(columns)
        fold_model.fit(X_fold_fit[columns], y_fold_fit)

        fold_predictions["model_score"] = fold_model.predict_proba(
            X_fold_valid[columns]
        )[:, 1]

        result = evaluate_top_share(
            fold_predictions,
            TARGET,
            fold_predictions["model_score"],
            share=TOP_SHARE,
            method_name=version_name
        )

        result["Фолд"] = fold
        cv_results.append(result)

cv_results_df = pd.DataFrame(cv_results).rename(
    columns={"Метод": "Версия"}
)

In [ ]:
cv_results_df[
    [
        "Фолд",
        "Версия",
        "Нагрузка, %",
        "Событий в отборе",
        "Доля событий, %",
        "Recall, %",
        "Lift"
    ]
].round(2)

,Фолд,Версия,"Нагрузка, %",Событий в отборе,"Доля событий, %","Recall, %",Lift
0,1,v0 baseline,10.0,341,71.04,32.14,3.21
1,1,v1 extended,10.0,334,69.58,31.48,3.15
2,2,v0 baseline,10.0,349,72.71,32.86,3.29
3,2,v1 extended,10.0,345,71.88,32.49,3.25
4,3,v0 baseline,10.0,333,69.38,31.36,3.14
5,3,v1 extended,10.0,337,70.21,31.73,3.17
6,4,v0 baseline,10.0,334,69.58,31.45,3.15
7,4,v1 extended,10.0,338,70.42,31.83,3.18
8,5,v0 baseline,10.0,335,69.79,31.54,3.15
9,5,v1 extended,10.0,338,70.42,31.83,3.18


In [ ]:
cv_summary = (
    cv_results_df
    .groupby("Версия")
    .agg(
        Среднее_событий=("Событий в отборе", "mean"),
        Средний_recall=("Recall, %", "mean"),
        Средний_lift=("Lift", "mean"),
        Минимальный_recall=("Recall, %", "min"),
        Максимальный_recall=("Recall, %", "max")
    )
    .reset_index()
)

cv_summary.round(2)

,Версия,Среднее_событий,Средний_recall,Средний_lift,Минимальный_recall,Максимальный_recall
0,v0 baseline,338.4,31.87,3.19,31.36,32.86
1,v1 extended,338.4,31.87,3.19,31.48,32.49


In [ ]:
fold_delta = (
    cv_results_df
    .pivot(
        index="Фолд",
        columns="Версия",
        values="Событий в отборе"
    )
)

fold_delta["Разница"] = (
    fold_delta["v1 extended"] - fold_delta["v0 baseline"]
)

fold_delta

Версия,v0 baseline,v1 extended,Разница
Фолд,,,
1,341,334,-7
2,349,345,-4
3,333,337,4
4,334,338,4
5,335,338,3


## Вывод по версии v1

Добавление MAX_DELINQ и RECENT_DELINQ_FLAG не улучшило средний
результат по основному критерию — числу событий в отборе
при нагрузке 10%.

На одинаковой пятифолдовой кросс-валидации обе версии охватили
в среднем 338,4 события на фолд. Средний recall после округления
составил 31,87%, средний lift — 3,19.

Разница v1 относительно v0 по фолдам:
−7, −4, +4, +4 и +3 события.

Новые признаки являются преобразованиями истории платёжных
статусов, уже включённой в v0. Однако эксперимент не устанавливает
причину отсутствия прироста и не доказывает бесполезность этих
признаков для других моделей или условий.

Для продолжения проекта выбрана более простая версия v0.
Тестовая выборка не использовалась.